In [ ]:
import pandas as pd
import sqlite3

from adler.objectdata.AdlerData import AdlerData
from adler.objectdata.AdlerPlanetoid import AdlerPlanetoid
from adler.utilities.tests_utilities import get_test_data_filepath

This is a quick notebook demonstrating how Adler's calculated values can be stored and then retrieved for later.

First, let's make our AdlerPlanetoid object. In this case, we're populating it from a testing SQL database.

In [ ]:
ssoid = "8268570668335894776"
test_db_path = get_test_data_filepath("testing_database.db")
test_planetoid = AdlerPlanetoid.construct_from_SQL(ssoid, test_db_path, filter_list=["g", "r"])

Now let's make up some pretend Adler calculated values, and populate the AdlerData object stored in AdlerPlanetoid.

In [ ]:
# TODO: need to sort out difference between modelId and model_name...
test_planetoid.AdlerData.modelId = "HG12_Pen16_fit1"

In [ ]:
g_model_1 = {
    "model_name": "HG12_Pen16",
    "phaseAngle_min": 31.0,
    "phaseAngle_range": 32.0,
    "nobs": 33,
    "arc": 34.0,
    "H": 35.0,
    "H_err": 36.0,
    "phase_parameter_1": 37.0,
    "phase_parameter_1_err": 38.0,
}

r_model_2 = {
    "model_name": "HG12_Pen16",
    "phaseAngle_min": 41.0,
    "phaseAngle_range": 42.0,
    "nobs": 43,
    "arc": 44.0,
    "H": 45.0,
    "H_err": 46.0,
    "phase_parameter_1": 47.0,
    "phase_parameter_1_err": 48.0,
    "phase_parameter_2": 49.0,
    "phase_parameter_2_err": 50.0,
}

In [ ]:
test_planetoid.AdlerData.populate_phase_parameters("g", **g_model_1)
test_planetoid.AdlerData.populate_phase_parameters("r", **r_model_2)

Now we can write these out.

In [ ]:
database_filepath = "./gen_test_data/example_AdlerData_database.db"

In [ ]:
# test_planetoid.AdlerData.__dict__
test_planetoid.AdlerData

In [ ]:
test_planetoid.AdlerData.write_to_database(database_filepath, write_model_data=True)

We'll use Pandas to look at what we just wrote out.

In [ ]:
cnx = sqlite3.connect(database_filepath)
example_query = "SELECT * FROM sqlite_schema"
pd.read_sql_query(example_query, cnx)

In [ ]:
pd.read_sql("SELECT * from AdlerData", cnx)

In [ ]:
pd.read_sql("SELECT * from FilterDependentAdler", cnx)

In [ ]:
pd.read_sql("SELECT * from PhaseModelDependentAdler", cnx)

Note that write_row_to_database() method always appends. So:

In [ ]:
# TODO: no longer appends, updates! need to change modelId or something to make append work?

In [ ]:
test_planetoid.AdlerData.write_to_database(database_filepath, write_model_data=True)

In [ ]:
cnx = sqlite3.connect(database_filepath)
pd.read_sql("SELECT * from AdlerData", cnx)

Now we have added two rows.

So perhaps we have an AdlerPlanetoid object and this time, we want to load in some previously calculated values for comparison. This is extremely easy. We'll do it on the AdlerPlanetoid object we already made.

In [ ]:
test_planetoid.attach_previous_adler_data(database_filepath)

This can be more easily accessed and read:

In [ ]:
test_planetoid.PreviousAdlerData.print_data()

In [ ]:
test_planetoid.PreviousAdlerData.get_phase_parameters_in_filter("g", "HG12_Pen16").__dict__

Or, if you don't want to work with an existing AdlerPlanetoid object, you can directly populate an AdlerData object from a database.

In [ ]:
adler_data_object = AdlerData(ssoid, ["g", "r"])
adler_data_object.populate_from_database(database_filepath)

In [ ]:
adler_data_object.print_data()

In [ ]:
adler_data_object.get_phase_parameters_in_filter("g", "HG12_Pen16").__dict__